# 00 — Environment Setup (AMD MI300X)

Run this notebook **once** on the AMD cloud notebook before the demo.
It installs dependencies, starts the two backing services (vLLM + Qdrant),
ingests the mock sanctions list, and health-checks everything.

Order matters — run cells top to bottom. Every check cell prints ✅ or a
fix-it hint. The pipeline also runs WITHOUT vLLM/Qdrant (mock fallbacks),
so a failed service is degraded, never fatal.

When done, open **01_demo.ipynb**.

In [ ]:
# 1) Python dependencies
%pip install -q streamlit langgraph openai pytesseract pillow \
    "qdrant-client[fastembed]" networkx requests pandas
print("✅ Python packages installed")

In [ ]:
# 2) Tesseract OCR — English + Hindi (Devanagari) for Aadhaar cards
!sudo apt-get update -qq && sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-hin
!tesseract --version | head -1

## 3) vLLM — Llama-3-8B-Instruct on the MI300X

Start vLLM in a **separate terminal** (it must keep running during the demo):

```bash
docker run --device=/dev/kfd --device=/dev/dri \
  --group-add video --ipc=host --shm-size 16G \
  -p 8000:8000 -v $HOME/models:/models \
  vllm/vllm-openai-rocm:latest \
  --model meta-llama/Meta-Llama-3-8B-Instruct
```

⚠️ If it comes up on a port other than 8000, change `VLLM_URL` in
`config.py` — that is the ONLY environment-specific value in the project.

Then run the health check below (model load can take a few minutes).

In [ ]:
# 3b) vLLM health check
import requests
from config import VLLM_API_BASE, VLLM_METRICS_URL

try:
    r = requests.get(f"{VLLM_API_BASE}/models", timeout=5)
    print("✅ vLLM is up — serving:", [m["id"] for m in r.json()["data"]])
    m = requests.get(VLLM_METRICS_URL, timeout=5)
    print("✅ /metrics live —", len(m.text.splitlines()), "telemetry lines for the dashboard")
except Exception as e:
    print("❌ vLLM not reachable:", e)
    print("   → start the docker container above, or fix VLLM_URL in config.py.")
    print("   The pipeline still runs in degraded mock mode without it.")

## 4) Qdrant — vector DB for sanctions screening

Start in another terminal (or background it):

```bash
docker run -d -p 6333:6333 qdrant/qdrant
```

In [ ]:
# 4b) Ingest the 55-entry mock sanctions list into Qdrant
from tools import setup_sanctions_collection

ok = setup_sanctions_collection()
if not ok:
    print("⚠️ Qdrant ingest failed — screening will use the deterministic")
    print("   local fuzzy-scan fallback instead. The demo still works.")

In [ ]:
# 5) Screening smoke test — should score ≈0.81 (ambiguous band)
from tools import query_sanctions_db

hits = query_sanctions_db("Vikram Malhotra")
for h in hits[:3]:
    print(f"{h['match_score']:.2f}  {h['matched_name']}  ({h['list_source']})")
assert hits and 0.60 <= hits[0]["match_score"] < 0.85, "expected an AMBIGUOUS-band match"
print("✅ screening calibration correct — self-correction loop is armed")

In [ ]:
# 6) ROCm telemetry check — feeds the live GPU dashboard (Tab 3)
!rocm-smi --showmeminfo vram --showuse || echo "rocm-smi not found — telemetry tab will show mock values"

In [ ]:
# 7) Full backend sanity check — both canonical customers, end to end
!python graph.py

## ✅ Setup complete

Continue with **01_demo.ipynb**, or go straight to the UI:

```bash
streamlit run app.py
```